In [10]:
!pip install pandas sqlalchemy psycopg2-binary matplotlib seaborn
# รัน Cell นี้ครั้งเดียวเพื่อติดตั้ง Library
%pip install matplotlib seaborn pandas sqlalchemy psycopg2-binary

zsh:1: command not found: pip

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# --- 1. Config Connection ---
# Username/Pass จากไฟล์ consumer.py หรือ .env
DB_USER = 'warehouse_admin'
DB_PASS = 'warehouse_password'
DB_HOST = 'localhost'
DB_PORT = '5433' # ย้ำ! ต้องใช้ 5433 ที่ Map ออกมาข้างนอก
DB_NAME = 'bitka_dw'

# สร้าง Connection String
connection_str = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(connection_str)

print("✅ Connecting to Data Warehouse...")
try:
    with engine.connect() as conn:
        print("🎉 Connection Successful! Ready to query.")
except Exception as e:
    print(f"❌ Connection Failed: {e}")

In [ ]:
# --- 2. Query Data: Price Trend ---
sql_price = """
SELECT 
    event_time, 
    symbol, 
    last_price, 
    volume_24h 
FROM fact_market_tickers 
WHERE symbol = 'BTC_THB' 
ORDER BY event_time DESC 
LIMIT 100
"""

# โหลดเข้า DataFrame
df_price = pd.read_sql(sql_price, engine)

# แปลงประเภทข้อมูลให้ถูกต้อง (เผื่อเป็น String/Decimal)
df_price['last_price'] = df_price['last_price'].astype(float)
df_price['event_time'] = pd.to_datetime(df_price['event_time'])

# แสดงข้อมูล 5 แถวแรก
display(df_price.head())

# --- 3. Visualization ---
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_price, x='event_time', y='last_price', marker='o')
plt.title('BTC_THB Real-time Price Movement')
plt.xlabel('Time')
plt.ylabel('Price (THB)')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
# --- 4. Query Data: Order Side Analysis ---
sql_orders = """
SELECT 
    side, 
    COUNT(*) as order_count,
    SUM(quantity * price) as total_value_thb
FROM fact_orders_created
WHERE symbol = 'BTC_THB'
GROUP BY side
"""

df_orders = pd.read_sql(sql_orders, engine)

# --- 5. Visualization: Pie Chart ---
plt.figure(figsize=(8, 8))
plt.pie(df_orders['order_count'], labels=df_orders['side'], autopct='%1.1f%%', colors=['#4CAF50', '#FF5252'])
plt.title('Buy vs Sell Order Distribution (BTC_THB)')
plt.show()

In [ ]:
sql_audit = """
SELECT 
    event_time, 
    actor_id, 
    action, 
    severity 
FROM fact_system_audit 
WHERE severity IN ('WARN', 'CRITICAL')
ORDER BY event_time DESC
"""

df_audit = pd.read_sql(sql_audit, engine)
display(df_audit)